# Fashion-MNIST ANN — Keras Sequential API

A complete teaching notebook for image classification using a moderate-depth Artificial Neural Network.

**Architecture:** `784 → 256 → Dropout → 128 → Dropout → 64 → 10`

Keras `Sequential` is used for model building. Each code cell is preceded by an explanation.

## 1. Import libraries
TensorFlow/Keras builds and trains the ANN. NumPy handles arrays, Matplotlib visualizes images/curves, and scikit-learn provides evaluation metrics.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow:", tf.__version__)

## 2. Load Fashion-MNIST
The dataset contains 60,000 training and 10,000 test grayscale images, each 28×28 pixels, belonging to 10 clothing classes.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test :", x_test.shape)
print("y_test :", y_test.shape)

## 3. Define class names
Labels are integers 0–9. This list lets us display meaningful clothing names.

In [ ]:
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

for i, name in enumerate(class_names):
    print(i, "->", name)

## 4. Visualize one image
Fashion-MNIST images are 28×28 grayscale images.

In [ ]:
plt.figure(figsize=(4,4))
plt.imshow(x_train[0], cmap="gray")
plt.title(f"Label: {y_train[0]} — {class_names[y_train[0]]}")
plt.axis("off")
plt.show()

## 5. Visualize multiple images
Viewing examples helps us understand the variation within the classes.

In [ ]:
plt.figure(figsize=(10,6))
for i in range(15):
    plt.subplot(3,5,i+1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(class_names[y_train[i]], fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Inspect pixel values
Original grayscale pixels are integers from 0 to 255.

In [ ]:
print("Minimum:", x_train.min())
print("Maximum:", x_train.max())

## 7. Normalize the images
Dividing by 255 converts pixels to the range 0–1, which gives the optimizer a more convenient numerical scale.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Minimum after normalization:", x_train.min())
print("Maximum after normalization:", x_train.max())

## 8. Understand the input
Each image has 28×28 = 784 pixels. The `Flatten` layer will convert this 2D image into a 784-element vector.

In [ ]:
print("Image shape:", x_train[0].shape)
print("Input features:", 28 * 28)

## 9. Define the ANN architecture

We will use:

```text
28×28 image
   ↓
Flatten → 784
   ↓
Dense(256, ReLU)
   ↓
Dropout(30%)
   ↓
Dense(128, ReLU)
   ↓
Dropout(30%)
   ↓
Dense(64, ReLU)
   ↓
Dense(10, Softmax)
```

This is an MLP/ANN because it uses fully connected Dense layers.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.summary()

## 10. Understand Dense(256)
The first Dense layer connects 784 input features to 256 neurons. Each neuron learns weights and a bias. ReLU introduces nonlinearity.

In [ ]:
params = 784 * 256 + 256
print("Dense(256) parameters:", params)

## 11. Understand Dropout
`Dropout(0.30)` randomly drops about 30% of activations during training. This is a regularization technique intended to reduce overfitting. It is disabled during inference.

In [ ]:
print("Dropout rate:", 0.30)

## 12. Understand the remaining layers
The network progressively transforms the representation: 256 → 128 → 64. The final Dense layer has 10 neurons because there are 10 classes.

In [ ]:
for layer in model.layers:
    print(layer.name, "->", layer.output.shape)

## 13. Count parameters
Keras calculates the number of trainable parameters for every layer and the complete model.

In [ ]:
print("Total parameters:", model.count_params())

## 14. Compile the model
Adam updates weights. Sparse categorical cross-entropy is appropriate because labels are integer class IDs 0–9. Accuracy tracks correct predictions.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

print("Model compiled.")

## 15. Train the ANN
An epoch is one pass through the training data. Batch size 128 means 128 samples are used for each parameter update. Ten percent of the training set is held out for validation.

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.10,
    verbose=1
)

## 16. Inspect training history
Keras stores loss and accuracy for both training and validation data.

In [ ]:
print(history.history.keys())

## 17. Plot loss
Training loss shows error on training data. Validation loss shows error on held-out validation data. A training loss that falls while validation loss rises can indicate overfitting.

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

## 18. Plot accuracy
Compare training and validation accuracy to understand learning and possible overfitting.

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

## 19. Evaluate on test data
The test set was not used to update the model. `evaluate` gives final loss and accuracy on unseen examples.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

## 20. Generate probabilities
The softmax output contains 10 probabilities for every image. The probabilities for each image sum to approximately 1.

In [ ]:
probabilities = model.predict(x_test, verbose=0)

print("Probability array shape:", probabilities.shape)
print("First image probabilities:", probabilities[0])

## 21. Convert probabilities to classes
`argmax` returns the index of the largest probability, giving the predicted class number.

In [ ]:
predicted_labels = np.argmax(probabilities, axis=1)

print("Predictions:", predicted_labels[:20])
print("Actual     :", y_test[:20])

## 22. Inspect one prediction
We compare the actual class, predicted class, and confidence of the predicted class.

In [ ]:
i = 0
pred = predicted_labels[i]
confidence = probabilities[i, pred]

print("Actual    :", class_names[y_test[i]])
print("Predicted :", class_names[pred])
print("Confidence:", f"{confidence:.2%}")

## 23. Visualize predictions
This helps connect numerical predictions to actual images.

In [ ]:
plt.figure(figsize=(12,8))
for i in range(20):
    plt.subplot(4,5,i+1)
    plt.imshow(x_test[i], cmap="gray")
    pred = predicted_labels[i]
    plt.title(f"A: {class_names[y_test[i]]}\nP: {class_names[pred]} ({probabilities[i,pred]:.0%})", fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 24. Build a confusion matrix
Rows represent actual classes and columns represent predicted classes. Diagonal values are correct predictions; off-diagonal values are errors.

In [ ]:
cm = confusion_matrix(y_test, predicted_labels)
print(cm)

## 25. Visualize the confusion matrix
The heatmap makes it easier to identify which clothing categories the ANN confuses.

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Fashion-MNIST Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 26. Classification report
Precision, recall, F1-score, and support provide class-level performance beyond overall accuracy.

In [ ]:
print(classification_report(
    y_test,
    predicted_labels,
    target_names=class_names
))

## 27. Find incorrect predictions
We locate examples where predicted and actual labels differ. These are useful for error analysis.

In [ ]:
incorrect = np.where(predicted_labels != y_test)[0]

print("Incorrect predictions:", len(incorrect))
print("First 10 incorrect indexes:", incorrect[:10])

## 28. Visualize incorrect predictions
Fashion-MNIST contains visually similar categories such as Shirt, T-shirt/top, Pullover, and Coat, so some errors are expected.

In [ ]:
plt.figure(figsize=(12,8))
for i, idx in enumerate(incorrect[:20]):
    plt.subplot(4,5,i+1)
    plt.imshow(x_test[idx], cmap="gray")
    actual = y_test[idx]
    pred = predicted_labels[idx]
    plt.title(f"A: {class_names[actual]}\nP: {class_names[pred]}", fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 29. Save the trained model
The `.keras` format stores the model architecture and learned weights so the model can be reused later.

In [ ]:
model_path = "fashion_mnist_ann.keras"
model.save(model_path)
print("Saved:", model_path)

## 30. Load the saved model
We can restore the model without rebuilding the architecture manually.

In [ ]:
loaded_model = tf.keras.models.load_model(model_path)
loaded_model.summary()

## 31. Verify the loaded model
We make a prediction with the restored model to confirm that it retained the learned parameters.

In [ ]:
p = loaded_model.predict(x_test[:1], verbose=0)
pred = np.argmax(p[0])

print("Actual:", class_names[y_test[0]])
print("Prediction:", class_names[pred])
print("Confidence:", f"{p[0,pred]:.2%}")

# ANN vs CNN

This notebook is an **ANN/MLP**, not a CNN.

ANN:

`28×28 → Flatten → Dense → Dense → Dense → Output`

CNN:

`28×28 → Conv2D → Pooling → Conv2D → Flatten → Dense → Output`

The ANN discards the 2D spatial arrangement when it flattens the image. CNNs preserve and exploit local spatial patterns, which is why CNNs are the natural next step for image classification.

# Key concepts

- **Dense:** fully connected layer.
- **ReLU:** nonlinear activation, `max(0,x)`.
- **Dropout:** regularization that randomly drops activations during training.
- **Softmax:** converts 10 output scores into class probabilities.
- **Loss:** measures prediction error.
- **Adam:** optimization algorithm that updates weights.
- **Epoch:** one complete pass through the training set.
- **Batch size:** number of samples used for one update.
- **Validation data:** used to monitor generalization during training.
- **Test data:** held back for final evaluation.

# Complete workflow

`Fashion-MNIST → Explore → Normalize → Sequential ANN → Compile → Train → Validate → Evaluate → Predict → Confusion Matrix → Error Analysis → Save Model`

This notebook is designed as a bridge from the earlier manually implemented TensorFlow MLP to practical Keras model development.